# 25 · Score v0, teste de partida, fila e rota (G-I5, G-I6, C16)

**Pontuação heurística, não previsão avaliada.** Fórmula e pesos em `src/wastestack/motor.py` e `config/motor_cenarios.json` (três cenários × morro/planície/indefinido; tudo provisório e editável).

**Teste de partida:** componentes calculados só com o 156 até ago/2025; alvo = chamados localizados de jan–mai/2026. A antiga referência de 35% era de **ruas**; aqui ela é refeita com a deduplicação atual e comparada também pelo mesmo km de via.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "src" / "wastestack").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
from wastestack.arquivos import ler
from wastestack.config import SILVER, GOLD

In [2]:
from wastestack.gold import motor_score, teste_partida, fila_e_rotas
resumo = motor_score()
resumo

{
 "cenario": "equilibrado",
 "limiar": 50,
 "hexagonos_pontuados": 1514,
 "faixas": {
  "médio": 905,
  "baixo": 492,
  "alto": 117
 },
 "acima_do_limiar": 120,
 "acima_do_limiar_por_regime": {
  "morro": 80,
  "planicie": 29,
  "indefinido": 11
 },
 "score_medio_por_regime": {
  "indefinido": 26.9,
  "morro": 34.3,
  "planicie": 36.0
 }
}


In [3]:
teste = teste_partida()
{k: v for k, v in teste.items() if k not in ('tabela', 'auditoria_quartil')}

{
 "hexagonos_avaliados": 1514,
 "chamados_2026_localizados": 12267.0,
 "ja_tinham_chamado_na_rua_pct": 93.4
}


In [4]:
t = ler("motor_teste_partida", camada=GOLD)
print(t[["criterio", "unidade", "unidades_no_topo", "km_de_via_no_topo", "pct_chamados_2026_capturados"]].to_string(index=False))
print()
print(ler("motor_auditoria_quartil", camada=GOLD).drop(columns=["uso", "fonte_dado"]).round(1).to_string(index=False))
fila = fila_e_rotas()
fila

                                                       criterio              unidade  unidades_no_topo  km_de_via_no_topo  pct_chamados_2026_capturados
                                 score v0 — cenário equilibrado             hexágono               151              479.8                          22.4
                                score v0 — cenário mais_demanda             hexágono               151              486.9                          25.7
                                  score v0 — cenário mais_risco             hexágono               151              429.8                          18.0
            histórico simples (total de chamados 2024–ago/2025)             hexágono               151              480.6                          28.8
                                               acaso (esperado)             hexágono               151                NaN                          10.0
ruas com mais histórico (todas as demandas, localizadas ou não) rua (texto + bairro)    

## Limites e cuidados

- **Resultado registrado sem enfeite:** nos 10% de hexágonos de maior score, o cenário equilibrado captura menos chamados de 2026 do que o histórico simples. É o esperado de um score que troca parte da demanda por risco estrutural; e mostra que o score **não é** um previsor de chamados.
- Chamado não é lixo: capturar chamados mede concordância com o 156, que subnotifica onde a carência é maior.
- A rota é vizinho mais próximo em **linha reta**; não é rota viária otimizada.